# 50 - Encode the scaled corpus with Nomic

Same structure as notebook 45/48/49, `nomic-ai/nomic-embed-text-v1.5` instead, same settings as `05_baseline_nomic.ipynb`: requires `trust_remote_code=True`, `normalize_embeddings=True` (cosine similarity), and a `search_document: ` prefix prepended to every document text (Nomic's asymmetric encoding convention, documents get `search_document:`, queries get `search_query:` at retrieval time).

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

RESULT_DIR = Path("result/50_encode_nomic_scaled")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RESULT_DIR / "company_embeddings_checkpoint.npy"
FINAL_PATH = RESULT_DIR / "company_embeddings.npy"
CHUNK_SIZE = 20_000

combined = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")
doc_texts = [f"search_document: {t}" for t in combined["rich_text"].tolist()]
print(f"[Load] Companies to encode: {len(doc_texts):,}")

print(f"[GPU] CUDA available : {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"[GPU] Device : {torch.cuda.get_device_name(0)}")

print("[Encode] Loading Nomic (nomic-ai/nomic-embed-text-v1.5)...")
t0 = time.time()
# Try local cache first with HF_HUB_OFFLINE=1 -- without this, sentence-transformers still makes a
# network call to verify the cache is current even when the model is fully cached, and on this
# cluster that unauthenticated call can hang long enough to burn the entire 30-min job with zero
# encoding progress (this is exactly what happened here previously). Fall back to network only if
# the cache turns out to be incomplete.
os.environ["HF_HUB_OFFLINE"] = "1"
try:
    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", device=DEVICE, trust_remote_code=True)
    print("[Encode] Loaded from local cache -- skipped Hugging Face Hub network calls")
except Exception as e:
    print(f"[Encode] Not fully cached locally yet ({type(e).__name__}) -- retrying with network access (this will be slower)")
    os.environ.pop("HF_HUB_OFFLINE", None)
    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", device=DEVICE, trust_remote_code=True)
print(f"[Encode] Model loaded in {time.time()-t0:.1f}s on {model.device}")
batch_size = 256 if DEVICE == "cuda" else 64
print(f"[Encode] Batch size : {batch_size}")

In [ ]:
CHUNKS_DIR = RESULT_DIR / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)


def atomic_save_npy(arr, path):
    """Write to a temp file then atomically rename -- a plain np.save() left a truncated,
    corrupted checkpoint on notebook 51 when a job was killed mid-write. Same fix applied here."""
    p = Path(path)
    tmp_path = p.with_suffix(".tmp" + p.suffix)
    np.save(tmp_path, arr)
    os.replace(tmp_path, path)


def get_chunk_files():
    return sorted(CHUNKS_DIR.glob("chunk_*.npy"), key=lambda p: int(p.stem.split("_")[1]))


if FINAL_PATH.exists() and np.load(FINAL_PATH, mmap_mode="r").shape[0] == len(doc_texts):
    print("[Encode] Final embeddings already on disk -- skipping")
    embeddings = np.load(FINAL_PATH)
else:
    chunk_files = get_chunk_files()
    start = sum(np.load(f, mmap_mode="r").shape[0] for f in chunk_files)

    # One-time migration: an older run may have left a single ever-growing checkpoint file
    # (the pattern that triggered a bwUniCluster high-I/O warning -- 746GB written against only
    # 43GB of actual data, since it rewrote the full accumulated array on every chunk). Fold
    # whatever it already has into the new per-chunk format once, instead of re-encoding it.
    if CHECKPOINT_PATH.exists() and start == 0:
        legacy = np.load(CHECKPOINT_PATH)
        print(f"[Encode] Migrating legacy checkpoint ({legacy.shape[0]:,} rows) into per-chunk format...")
        atomic_save_npy(legacy, CHUNKS_DIR / f"chunk_{0:09d}.npy")
        start = legacy.shape[0]
        CHECKPOINT_PATH.unlink()
        chunk_files = get_chunk_files()

    if start:
        print(f"[Encode] Resuming -- {start:,}/{len(doc_texts):,} already encoded across {len(chunk_files)} chunk files")

    t0 = time.time()
    for chunk_start in range(start, len(doc_texts), CHUNK_SIZE):
        chunk_texts = doc_texts[chunk_start:chunk_start + CHUNK_SIZE]
        chunk_embs = model.encode(
            chunk_texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,  # REQUIRED -- Nomic uses cosine similarity
        )
        # Save ONLY this chunk as its own small file -- never rewrite everything already saved.
        atomic_save_npy(chunk_embs, CHUNKS_DIR / f"chunk_{chunk_start:09d}.npy")
        done_so_far = chunk_start + len(chunk_texts)
        elapsed = time.time() - t0
        print(f"[Encode] {done_so_far:,}/{len(doc_texts):,} encoded ({elapsed/60:.1f} min elapsed)")

    # Assemble the final array exactly once, only now that every chunk is done.
    chunk_files = get_chunk_files()
    embeddings = np.concatenate([np.load(f) for f in chunk_files], axis=0)
    atomic_save_npy(embeddings, FINAL_PATH)
    for f in chunk_files:
        f.unlink()
    print(f"[Encode] Done. Embeddings shape: {embeddings.shape}")
    print(f"[Encode] Saved -> {FINAL_PATH}")